In [6]:
import os
import json
import glob

import numpy as np
import pandas as pd

In [7]:
runs = sorted(glob.glob('gp_data_l40s/*/*/*'))
runs = [run for run in runs if 'results.csv' not in run]
runs

['gp_data_l40s/GP_BDY_L40S/donut/seed_1',
 'gp_data_l40s/GP_BDY_L40S/donut/seed_2',
 'gp_data_l40s/GP_BDY_L40S/donut/seed_3',
 'gp_data_l40s/GP_BDY_L40S/donut/seed_4',
 'gp_data_l40s/GP_BDY_L40S/donut/seed_5',
 'gp_data_l40s/GP_BDY_L40S/hopf/seed_1',
 'gp_data_l40s/GP_BDY_L40S/hopf/seed_2',
 'gp_data_l40s/GP_BDY_L40S/hopf/seed_3',
 'gp_data_l40s/GP_BDY_L40S/hopf/seed_4',
 'gp_data_l40s/GP_BDY_L40S/hopf/seed_5',
 'gp_data_l40s/GP_BDY_L40S/planewave/seed_1',
 'gp_data_l40s/GP_BDY_L40S/planewave/seed_2',
 'gp_data_l40s/GP_BDY_L40S/planewave/seed_3',
 'gp_data_l40s/GP_BDY_L40S/planewave/seed_4',
 'gp_data_l40s/GP_BDY_L40S/planewave/seed_5',
 'gp_data_l40s/GP_BDY_L40S/random/seed_1',
 'gp_data_l40s/GP_BDY_L40S/random/seed_2',
 'gp_data_l40s/GP_BDY_L40S/random/seed_3',
 'gp_data_l40s/GP_BDY_L40S/random/seed_4',
 'gp_data_l40s/GP_BDY_L40S/random/seed_5',
 'gp_data_l40s/GP_NBDY_L40S/donut/seed_1',
 'gp_data_l40s/GP_NBDY_L40S/donut/seed_2',
 'gp_data_l40s/GP_NBDY_L40S/donut/seed_3',
 'gp_data_l

In [8]:
df = pd.DataFrame(columns=['add_bc', 'soln_name', 'seed', 'min_best_val_rel_err', 'time_to_min_best_val_rel_err', 'time_to_5p_best_val_rel_err'])

In [9]:
for run in runs:
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    log = log.sort_values('epoch').reset_index(drop=True)
    log['best_val_rel_err'] = log['best_val_rel_err'] * 100.0
    log = log.round(2)
    
    info = {}
    
    split = run.split(os.path.sep)
    
    info['add_bc'] = True if '_BDY' in split[1] else False
    info['soln_name'] = split[2]
    info['seed'] = split[3].replace('seed_', '')
    
    # work in fraction-space throughout; only convert to percent for the final stored value
    min_best_val_rel_err_frac = log['best_val_rel_err'].min()
    info['min_best_val_rel_err'] = min_best_val_rel_err_frac
    
    # epoch where we reach minimum validation relative L2 err
    min_best_val_rel_err_epoch = int(log[log['best_val_rel_err'] <= min_best_val_rel_err_frac].iloc[0]['epoch'])
    info['time_to_min_best_val_rel_err'] = log[log['epoch'] <= min_best_val_rel_err_epoch]['time_elapsed'].iloc[-1].item()
    
    # epoch where we first reach 5% relative err (0.05 in fraction-space)
    try:
        first_5p_epoch = int(log[log['best_val_rel_err'] < 5.0].iloc[0]['epoch'])
        info['time_to_5p_best_val_rel_err'] = log[log['epoch'] <= first_5p_epoch]['time_elapsed'].iloc[-1].item()
    except IndexError:
        info['time_to_5p_best_val_rel_err'] = np.nan
    
    df.loc[len(df)] = info

In [10]:
df['seed'] = df['seed'].astype(float)

results = df[[
    'soln_name',
    'time_to_5p_best_val_rel_err',
    'time_to_min_best_val_rel_err',
    'min_best_val_rel_err',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_5p_best_val_rel_err': 'Time to <5% RL2E. (s)',
    'time_to_min_best_val_rel_err': 'Time to min. RL2E. (s)',
    'min_best_val_rel_err': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative err columns
        display_results[col] = mean.map('{:.2f}'.format) + ' ± ' + sem.map('{:.2f}'.format)

display_results = pd.DataFrame(display_results).reset_index()
display_results['Solution'] = display_results['Solution'].map(
    {'random': 4, 'planewave': 1, 'donut': 2, 'hopf': 3}
)
display_results = display_results.sort_values(by=['Solution', 'w/ BC'], ascending=[True, False])
display_results = display_results[['Solution', 'w/ BC', 'Min. RL2E. (%)', 'Time to <5% RL2E. (s)', 'Time to min. RL2E. (s)']].reset_index(drop=True)

In [11]:
display_results

,Solution,w/ BC,Min. RL2E. (%),Time to <5% RL2E. (s),Time to min. RL2E. (s)
0,1,True,31.17 ± 1.24,NaN,4193.6 ± 972.0
1,1,False,57.32 ± 1.30,NaN,392.9 ± 88.5
2,2,True,68802413822.68 ± 39751822875.60,NaN,1295.8 ± 1294.5
3,2,False,40.01 ± 1.20,NaN,326.8 ± 101.0
4,3,True,473.16 ± 13.51,NaN,2831.5 ± 842.5
5,3,False,16.10 ± 0.74,NaN,510.1 ± 36.7
6,4,True,1618.54 ± 146.11,NaN,3610.8 ± 684.7
7,4,False,30.32 ± 2.73,NaN,423.9 ± 88.7


In [14]:
print(display_results.to_markdown(index=False))

|   Solution | w/ BC   | Min. RL2E. (%)                  |   Time to <5% RL2E. (s) | Time to min. RL2E. (s)   |
|-----------:|:--------|:--------------------------------|------------------------:|:-------------------------|
|          1 | True    | 31.17 ± 1.24                    |                     nan | 4193.6 ± 972.0           |
|          1 | False   | 57.32 ± 1.30                    |                     nan | 392.9 ± 88.5             |
|          2 | True    | 68802413822.68 ± 39751822875.60 |                     nan | 1295.8 ± 1294.5          |
|          2 | False   | 40.01 ± 1.20                    |                     nan | 326.8 ± 101.0            |
|          3 | True    | 473.16 ± 13.51                  |                     nan | 2831.5 ± 842.5           |
|          3 | False   | 16.10 ± 0.74                    |                     nan | 510.1 ± 36.7             |
|          4 | True    | 1618.54 ± 146.11                |                     nan | 3610.8 ± 684.7     